# Sales Prediction Using Python
**OASIS INFOBYTE — Data Science Internship, Task 2**  
**Student:** MAYUR R SHETTY

## Objective
Predict product sales from TV, Radio, and Newspaper advertising budgets. Compare Linear Regression with Random Forest and evaluate with MAE, RMSE, and R².

## 1. Import libraries and load data
The notebook downloads the classic ISLR Advertising CSV if it is not already in `data/`. If running in an offline environment, place `Advertising.csv` in the `data/` folder first.

In [ ]:
from pathlib import Path
import urllib.request, json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
DATA_DIR, OUTPUT_DIR = ROOT / "data", ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_FILE = DATA_DIR / "Advertising.csv"
urls = ["https://www.statlearning.com/s/Advertising.csv", "https://raw.githubusercontent.com/nguyen-toan/ISLR/master/dataset/Advertising.csv"]

if not DATA_FILE.exists():
    errors = []
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=25) as response:
                DATA_FILE.write_bytes(response.read())
            break
        except Exception as exc:
            errors.append(str(exc))
    if not DATA_FILE.exists():
        raise RuntimeError("Download Advertising.csv manually from https://www.statlearning.com/s/Advertising.csv and place it in data/. " + " | ".join(errors))

data = pd.read_csv(DATA_FILE)
data.columns = [str(c).strip().lower() for c in data.columns]
data = data.loc[:, ~data.columns.str.startswith("unnamed")]
data = data.drop(columns=["x", "x1", "index"], errors="ignore")
required = ["tv", "radio", "newspaper", "sales"]
data = data[required].apply(pd.to_numeric, errors="coerce").dropna().drop_duplicates().reset_index(drop=True)
display(data.head())
print("Dataset shape:", data.shape)

## 2. Data inspection
Review column types, missing values, and summary statistics.

In [ ]:
print(data.info())
print("\\nMissing values:")
display(data.isna().sum().to_frame("missing values"))
print("\\nDescriptive statistics:")
display(data.describe().T)

## 3. Exploratory Data Analysis
The scatter matrix acts as a pairplot and shows relationships among all variables.

In [ ]:
scatter_matrix(data, figsize=(10, 10), diagonal="hist", alpha=0.65)
plt.suptitle("Advertising Dataset — Pairwise Relationships", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pairplot.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
for channel in ["tv", "radio", "newspaper"]:
    plt.figure(figsize=(7, 5))
    plt.scatter(data[channel], data["sales"], alpha=0.7)
    slope, intercept = np.polyfit(data[channel], data["sales"], 1)
    x_line = np.linspace(data[channel].min(), data[channel].max(), 100)
    plt.plot(x_line, slope*x_line + intercept)
    plt.title(f"Sales vs {channel.title()} Advertising")
    plt.xlabel(f"{channel.title()} advertising budget (thousands)")
    plt.ylabel("Sales (thousands of units)")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"sales_vs_{channel}.png", dpi=160)
    plt.show()

In [ ]:
corr = data.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=25, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
ax.set_title("Correlation Heatmap")
fig.colorbar(im, ax=ax, label="Correlation")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=160)
plt.show()
display(corr["sales"].sort_values(ascending=False))

## 4. Train/test split
Separate the three advertising channels as input features and Sales as the target. Use an 80/20 split with a fixed random seed for reproducibility.

In [ ]:
X = data[["tv", "radio", "newspaper"]]
y = data["sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))

## 5. Train regression models
Linear Regression is the required baseline. Random Forest Regressor provides a non-linear comparison.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=400, random_state=42, min_samples_leaf=2
    )
}
fitted, predictions, metrics = {}, {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted[name], predictions[name] = model, pred
    metrics[name] = {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    }
metric_df = pd.DataFrame(metrics).T
display(metric_df.round(4))

## 6. Model evaluation
MAE and RMSE measure prediction error (lower is better). R² measures explained variance (closer to 1 is generally better).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    metric_df[metric].plot(kind="bar", ax=ax, title=metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
fig.suptitle("Regression Model Comparison")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "model_comparison.png", dpi=160)
plt.show()

In [ ]:
best_name = max(metrics, key=lambda name: (metrics[name]["R2"], -metrics[name]["RMSE"]))
best_model, best_pred = fitted[best_name], predictions[best_name]
plt.figure(figsize=(6, 5))
plt.scatter(y_test, best_pred, alpha=0.75)
low, high = min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.title(f"Actual vs Predicted — {best_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=160)
plt.show()
print("Selected model:", best_name)
print(metric_df.loc[best_name])

## 7. Residual analysis
Residual = actual sales − predicted sales. Ideally, residuals are randomly scattered around zero without a clear curve or funnel pattern.

In [ ]:
residuals = y_test.to_numpy() - best_pred
plt.figure(figsize=(7, 5))
plt.scatter(best_pred, residuals, alpha=0.75)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted sales")
plt.ylabel("Residual (actual - predicted)")
plt.title(f"Residual Plot — {best_name}")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "residual_plot.png", dpi=160)
plt.show()

## 8. Advertising channel impact
Use the multiple Linear Regression coefficients to estimate the marginal association of each channel with sales while holding the other channels constant. Coefficients are comparable here because the three budget features share the same units. This is not proof of causation.

In [ ]:
linear_model = fitted["Linear Regression"]
coef_impact = pd.Series(linear_model.coef_, index=X.columns)
display(coef_impact.sort_values(key=abs, ascending=False).to_frame("linear regression coefficient"))
strongest = coef_impact.abs().idxmax()
print("Largest absolute coefficient:", strongest)
print("Correlation with sales:")
display(data.corr(numeric_only=True)["sales"].drop("sales").sort_values(ascending=False))

## 9. Save results and conclusion

In [ ]:
result = {
    "student": "MAYUR R SHETTY",
    "model_metrics": metric_df.to_dict(orient="index"),
    "selected_model": best_name,
    "linear_regression_coefficients": coef_impact.to_dict(),
    "most_influential_channel_by_absolute_linear_coefficient": strongest
}
(OUTPUT_DIR / "results.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
joblib.dump(best_model, OUTPUT_DIR / "best_sales_model.joblib")
print("Saved results and trained model in:", OUTPUT_DIR.resolve())

### Conclusion
The project compares Linear Regression and Random Forest using MAE, RMSE, and R². Use the actual results displayed above to identify the best-performing model. The coefficient analysis reports the channel with the largest absolute marginal coefficient; compare this with the correlation results and explain why correlation and multivariable regression coefficients can differ.

### Limitations
This classic dataset contains 200 markets and is educational. It does not establish causal effects or guarantee performance for modern advertising campaigns.

**Submission reminder:** Run every cell, inspect all output files, add your own screenshots, and record your own demo video.